# Day 1 — Gemini API Quickstart

Verifies that the development environment is fully working before building the RAG pipeline.
Two things must succeed: **text generation** with `gemini-2.5-flash` and **embeddings** with
`gemini-embedding-001`.

This notebook uses the **free Gemini API path** (`GEMINI_API_KEY` from `.env`).  
The Vertex AI verification path is Day 4.

**SDK:** `google-genai` (the current SDK — `google-generativeai` is deprecated as of 2025)

## 1. Setup

In [1]:
import os
import time
import numpy as np
from google import genai
from google.genai import errors as genai_errors
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

api_key = os.environ.get("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY not found — check your .env file"

client = genai.Client(api_key=api_key)

GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

def with_retry(fn, retries=4, base_delay=5):
    """Retry fn on 503 UNAVAILABLE with exponential backoff."""
    for attempt in range(retries):
        try:
            return fn()
        except genai_errors.ServerError as e:
            if attempt == retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"503 UNAVAILABLE — retrying in {delay}s (attempt {attempt + 1}/{retries})...")
            time.sleep(delay)

print(f"API key loaded : {api_key[:8]}...")
print(f"Generation     : {GENERATION_MODEL}")
print(f"Embedding      : {EMBEDDING_MODEL}")

API key loaded : AIzaSyDJ...
Generation     : gemini-2.5-flash
Embedding      : gemini-embedding-001


## 2. Text Generation

A single `generate_content` call to confirm the model is reachable and the key has quota.

In [2]:
response = with_retry(lambda: client.models.generate_content(
    model=GENERATION_MODEL,
    contents="What is Vertex AI in one sentence?",
))

print("Response:")
print(response.text)

Response:
Vertex AI is Google Cloud's unified, end-to-end machine learning platform for building, deploying, and managing ML models across their entire lifecycle.


## 3. Embeddings

Embed a sample string and confirm the output shape. `task_type="RETRIEVAL_QUERY"` is the correct
type for user queries; corpus chunks use `RETRIEVAL_DOCUMENT`.

In [3]:
result = with_retry(lambda: client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="What is Vertex AI?",
))

vector = np.array(result.embeddings[0].values)
print(f"Embedding shape : {vector.shape}")
print(f"Vector norm     : {np.linalg.norm(vector):.4f}")
print(f"First 5 values  : {vector[:5]}")

Embedding shape : (3072,)
Vector norm     : 1.0000
First 5 values  : [-0.02307934  0.01064724 -0.00395659 -0.07197455 -0.05342148]


## 4. Cosine Similarity Smoke Test

Embeds two sentences and computes their cosine similarity — a sanity check that the
embeddings are semantically meaningful before we build the full retrieval pipeline.

In [4]:
def embed(text: str) -> np.ndarray:
    r = with_retry(lambda: client.models.embed_content(model=EMBEDDING_MODEL, contents=text))
    return np.array(r.embeddings[0].values)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v1 = embed("Vertex AI is Google Cloud's platform for machine learning.")
v2 = embed("Google Cloud offers a managed ML platform called Vertex AI.")
v3 = embed("The weather in Dallas is hot in summer.")

print(f"Similar sentences  : {cosine_similarity(v1, v2):.4f}  (expect high, ~0.9+)")
print(f"Unrelated sentence : {cosine_similarity(v1, v3):.4f}  (expect low, ~0.5-)")

Similar sentences  : 0.8812  (expect high, ~0.9+)
Unrelated sentence : 0.4972  (expect low, ~0.5-)


## Summary

If all cells ran without errors:

- ✅ `GEMINI_API_KEY` is valid and has quota  
- ✅ `gemini-2.5-flash` generates coherent text  
- ✅ `gemini-embedding-001` returns a `(3072,)` unit vector  
- ✅ Cosine similarity distinguishes related vs. unrelated sentences  

**Day 1 complete. Ready to build the RAG pipeline (Day 2).**